In [ ]:
!pip install transformers torch matplotlib seaborn

## Parte 1 - Transformers
Los transformers son muy sencillos de utilizar en código, con solo 3 líneas se puede hacer uno.

In [ ]:
from transformers import AutoTokenizer

sentence = "El perro estaba durmiendo tranquilo"

tokenizer = AutoTokenizer.from_pretrained("bert-base-multilingual-cased")

tokens = tokenizer.tokenize(sentence)
token_ids = tokenizer.convert_tokens_to_ids(tokens)

print("Frase:", sentence)
print("Tokens:", tokens)
print("Token IDs:", token_ids)

Qué hace internamente?
- 1. Tokenización: Convierte el texto en tokens (palabras o subpalabras). Tokenización → texto → tokens.
- 2. Embeddings: Transforma los tokens en vectores numéricos.
- 3. Capa de atención: Permite al modelo enfocarse en partes relevantes del texto. Calcula relaciones entre todos los tokens. El paper Attention is All You Need explica esta parte, es la más diferenciadora e innovadora. -> Es la principal diferencia entre transformers y otros modelos, el cómo obtienen el contexto.
- 4. Capa de salida: Produce la predicción final (positivo o negativo). Puede ser softmax o sigmoid, aunque no está limitado a ser positivo/negativo, puede generar texto, traducir, resumir...

Es similar a una red neuronal clásica?
- Sí, pero con una arquitectura específica (transformer) y técnicas avanzadas como la atención.

Diferencias entre esta arquitectura y una red neuronal tradicional?
- 1. La arquitectura de DL NO tiene tokenización, los transformer trabajan con texto
- 2. La arquitectura de DL SÍ tiene embeddings, excepto en uso de imágenes
- 3. La arquitectura de DL NO tiene atención, DL suele tener convoluciones o recurrentes para obtener el contexto. Los transformer capturan el contexto viendo las relaciones globales de cada uno de los elementos del texto.
- 4. La arquitectura de DL SÍ tiene capa de salida, aunque en transformers suele ser común que sea texto en lugar de una clasificación.

## Parte 2 - Atención
La atención es lo más revolucionario del transformer, le permite conocer si una palabra es importante o no en el texto

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np
import matplotlib.pyplot as plt

model_name = "bert-base-multilingual-cased"
model = AutoModel.from_pretrained(model_name, output_attentions=True)
#Cargamos un modelo preentrenado para ver la atención en diferentes frases

In [ ]:
sentences = [ #Frases a usar con la palabra "banco"
    "Necesito dinero del banco",
    "Me siento en el banco del parque",
    "Vi un banco de peces en el mar"
]

In [ ]:
def analyze_attention(sentence, target_word="banco", layer=0): #función de ayuda para inspeccionar la frase
    inputs = tokenizer(sentence, return_tensors="pt")
    with torch.no_grad():
        outputs = model(**inputs)

    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    att = outputs.attentions[layer][0].mean(dim=0).cpu().numpy()

    print(f"\nFrase: {sentence}")
    print("Tokens:", tokens)

    idx_candidates = [i for i, t in enumerate(tokens) if target_word in t.lower()]
    if not idx_candidates:
        print(f"No se encontró '{target_word}' como token directo.")
        return tokens, att

    i = idx_candidates[0]
    row = att[i].copy()
    row[i] = 0.0

    valid = [(j, row[j]) for j in range(len(tokens)) if tokens[j] not in ["[CLS]", "[SEP]"]]
    top = sorted(valid, key=lambda x: x[1], reverse=True)[:3]
    print(f"Top atención desde '{tokens[i]}':", [(tokens[j], float(v)) for j, v in top])

    return tokens, att

In [ ]:
results = []
for s in sentences: #analizamos la atención por cada frase para ver si "banco" se considera importante y a qué otras palabras atiende
    tokens, att = analyze_attention(s, target_word="banco", layer=0)
    results.append((s, tokens, att))

In [ ]:
for s, tokens, att in results:
    plt.figure(figsize=(7,6))
    plt.imshow(att, cmap="viridis")
    plt.xticks(range(len(tokens)), tokens, rotation=90)
    plt.yticks(range(len(tokens)), tokens)
    plt.title(f"Atención - {s}")
    plt.colorbar()
    plt.tight_layout()
    plt.show()

## Ejercicio
- Crea otros dos ejemplos de frases con palabras ambiguas para ver si entiende la relación o no (cura, vela...)
- Anota las 2 palabras con las que tiene más atención la palabra ambigua
- ¿Por qué tiene relación con esa palabra?

## Ejercicio 2 - Respuestas
Los modelos tambien pueden generar respuestas tras conocer el contexto de lo que se les dice

In [ ]:
from transformers import pipeline

unmasker = pipeline("fill-mask", model="bert-base-multilingual-cased")

print(unmasker("Necesito dinero del [MASK]"))
print(unmasker("Me siento en el [MASK] del parque"))